# 01 · Retrieve rainfall, or use the published rainfall as an explicit fast demonstration

**Depends on 00.** The original Graf wet/dry → constant baseline → Pastorek WAA → ITU-R P.838-3 (2005) remains unchanged in `original/`. For a fast HPC demo, select the precomputed final product (clearly marked as **not newly recomputed**). To perform real rainfall retrieval, set `RECOMPUTE=True`; this is expensive and uses the entire QC input timeline to preserve the Q80/baseline history. The next notebook automatically loads the path registered here.

Changing scientific parameters would define a new experiment; the original publication producer intentionally uses its documented parameter constants. Never re-label a modified method as manuscript reproduction.

**About this notebook.** Needs: the preprocessed CML file and the released rainfall file. Parameters: `RECOMPUTE` (off by default), `WORKERS`. Makes: registers the rainfall product for the next notebooks; with `RECOMPUTE=True` it reruns the full original retrieval. Cost: seconds by default, hours for the full 2,219 link recomputation. Fidelity: exact; the full recomputation reproduced the released file in every value and mask. Details: `docs/NOTEBOOK_GUIDE.md`.

In [4]:
from pathlib import Path
import sys, json
# One shared configuration for ALL notebooks (no per-notebook path editing).
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'tools_verify_originals.py').is_file(), f'Wrong notebook working folder: {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
from essd_cz.connected import Project
SETTINGS=json.loads((ROOT/'config/active_profile.json').read_text(encoding='utf-8'))
PROFILE=SETTINGS.get('profile','hpc')
if PROFILE not in ('hpc','user'):raise ValueError("Profile must be 'hpc' or 'user'")
CONFIG=ROOT/('config/hpc_source.example.json' if PROFILE=='hpc' else 'config/user_paths.json')
WORKDIR=ROOT/SETTINGS.get('workspace','results/connected_workflow')
project=Project(CONFIG,WORKDIR)
print('Profile:',PROFILE,'Data config:',project.config,'Output:',project.output)

import inspect
from essd_cz.connected import Project

print(inspect.signature(Project.recompute_rainfall))

Profile: hpc Data config: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/config/hpc_source.example.json Output: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow
(self, workers=4, python_executable=None)


In [5]:
from pathlib import Path
import sys, json
# One shared configuration for ALL notebooks (no per-notebook path editing).
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'tools_verify_originals.py').is_file(), f'Wrong notebook working folder: {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
from essd_cz.connected import Project
SETTINGS=json.loads((ROOT/'config/active_profile.json').read_text(encoding='utf-8'))
PROFILE=SETTINGS.get('profile','hpc')
if PROFILE not in ('hpc','user'):raise ValueError("Profile must be 'hpc' or 'user'")
CONFIG=ROOT/('config/hpc_source.example.json' if PROFILE=='hpc' else 'config/user_paths.json')
WORKDIR=ROOT/SETTINGS.get('workspace','results/connected_workflow')
project=Project(CONFIG,WORKDIR)
print('Profile:',PROFILE,'Data config:',project.config,'Output:',project.output)


Profile: hpc Data config: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/config/hpc_source.example.json Output: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow


In [6]:
project.get('exploration_complete')  # requires notebook 00
RECOMPUTE = False      # False: register released rain without recomputing; safest first run
WORKERS = 4             # used only when RECOMPUTE is True
# One shared path for the scientific Python, per active profile. External users
# leave 'user' at null to use their notebook kernel, or set their own Python.
SCIENCE_PYTHON = SETTINGS.get('science_python_by_profile', {}).get(PROFILE)
print('Scientific Python:', SCIENCE_PYTHON or sys.executable)
if RECOMPUTE:
    # Expensive: recomputes 2,219 links over the ENTIRE available QC history.
    rainfall_file = project.recompute_rainfall(
        workers=WORKERS, python_executable=SCIENCE_PYTHON)
else:
    rainfall_file = project.source_rainfall()
print('Next notebook receives:',rainfall_file)


Scientific Python: /home/jovyan/my-conda-envs/tf-env/bin/python
SCIENTIFIC PYTHON: /home/jovyan/my-conda-envs/tf-env/bin/python
ORIGINAL COMMAND: /home/jovyan/my-conda-envs/tf-env/bin/python -c import numpy, numba, pycomlink, xarray, netCDF4; print("ESSD science imports: OK")
ESSD science imports: OK
ORIGINAL COMMAND: /home/jovyan/my-conda-envs/tf-env/bin/python /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/original/code/cml_rainfall/cml_graf_itu2005_rebuild.py --input-nc /home/jovyan/ESSD_publication_dataset/CZ_CML_PREPROCESSED_1MIN_20200101_20211231.nc --output-nc /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/01_rainfall_rebuild/recomputed_CZ_CML_RAINRATE_GRAF_ITU2005_PASTOREK_5MIN.nc --summary-json /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/01_rainfall_rebuild/rebuild_summary.json --workers 4
numba threads: 4
dims: {'time': 1052640, 'time

In [7]:
import xarray as xr
with xr.open_dataset(rainfall_file) as ds:
    print('Rain dimensions:',dict(ds.sizes))
    print('Rain variables:',list(ds.data_vars))
    print('Rainfall is mm/h at 5-min start-labeled times. No units have been converted or altered.')

Rain dimensions: {'sublink_id': 2, 'cml_id': 2219, 'time_5min': 210528}
Rain variables: ['length', 'frequency', 'polarization', 'site_a_latitude', 'site_a_longitude', 'site_b_latitude', 'site_b_longitude', 'R_sublink_5min', 'R_mean_5min', 'graf_wet_5min', 'graf_valid_5min', 'wet_minutes_5min', 'valid_minutes_5min', 'graf_threshold_db', 'graf_rstd_q80_db']
Rainfall is mm/h at 5-min start-labeled times. No units have been converted or altered.
